# Never Click "Refresh Schema" Again: Automating SQL Endpoint Sync in Microsoft Fabric

## The Problem

When you modify Delta tables in a Fabric Lakehouse (add columns, create new tables, change data types), the **SQL analytics endpoint does not update automatically**. The endpoint maintains its own metadata cache, and until that cache is refreshed, your new columns are invisible to anyone querying through SQL.

The manual fix? Navigate to the Lakehouse in the Fabric portal, open the SQL analytics endpoint, and click **"Refresh Schema."** On large tables, this takes 10 to 12 minutes and offers no progress feedback. Worse, it is not automatable: you cannot embed a UI click into a Data Pipeline or a scheduled notebook.

## The Solution

Microsoft provides an official REST API endpoint for on-demand metadata refresh:

```
POST /v1/workspaces/{workspaceId}/sqlEndpoints/{sqlEndpointId}/refreshMetadata
```

This notebook demonstrates the full workflow:

1. **Ingest** a 60M-row travel dataset into a partitioned Delta table
2. **Explore** the data with EDA queries
3. **Engineer features** by adding 12 new calculated columns to the Delta table
4. **Expose the problem**: the SQL endpoint still shows the original 30 columns
5. **Solve it programmatically**: trigger metadata refresh via the Fabric REST API
6. **Verify**: confirm all 42 columns are now visible in the SQL endpoint

## Demo Scenario: VoyageHub Travel Super-App

VoyageHub is a unified travel platform (flights, hotels, rides) with 60 million booking transactions spanning 2022 to 2025. The dataset includes 30 columns covering user demographics, booking details, financials, and engagement metrics.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration and Imports
# ---------------------------------------------------------------------------
import time
import json
import uuid
import re
import logging
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

# ---------------------------------------------------------------------------
# Notebook Configuration
# ---------------------------------------------------------------------------
DELTA_TABLE = "voyagehub_transactions"
SOURCE_PATH = "Files/voyagehub_dataset/"
PARTITION_COLS = ["booking_year", "booking_month"]

# Run metadata
RUN_ID = str(uuid.uuid4())[:8]
RUN_START = datetime.now(timezone.utc)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("voyagehub_blog")

print("=" * 70)
print("  VOYAGEHUB SQL ENDPOINT REFRESH NOTEBOOK")
print("=" * 70)
print(f"  Run ID       : {RUN_ID}")
print(f"  Start        : {RUN_START.isoformat()}")
print(f"  Source       : {SOURCE_PATH}")
print(f"  Delta Table  : {DELTA_TABLE}")
print(f"  Partitions   : {PARTITION_COLS}")
print("=" * 70)

---
## Phase 1: Data Ingestion

We start by reading all Parquet chunk files from the Lakehouse `Files/` area and writing them as a single, partitioned Delta table. The dataset was generated locally using `generate_voyagehub_data.py` and uploaded to the Lakehouse.

**Key decisions:**
- **Partitioning by `booking_year` and `booking_month`** for efficient time-range queries and OPTIMIZE operations
- **Overwrite mode** for idempotency: re-running this notebook will not create duplicate data
- Partition columns are derived from `booking_date` during ingestion (they do not exist in the source Parquet files)

In [ ]:
# ---------------------------------------------------------------------------
# Phase 1: Data Ingestion
# ---------------------------------------------------------------------------
ingest_start = time.time()

# Read all Parquet files from Lakehouse Files section
df_raw = spark.read.parquet(f"{SOURCE_PATH}*.parquet")
source_count = df_raw.count()
source_cols = len(df_raw.columns)
print(f"Source: {source_count:,} rows, {source_cols} columns")

# Add partition columns derived from booking_date
df_partitioned = (
    df_raw
    .withColumn("booking_year", F.year("booking_date"))
    .withColumn("booking_month", F.month("booking_date"))
)

# Write as partitioned Delta table (overwrite for idempotency)
(
    df_partitioned.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy(*PARTITION_COLS)
    .saveAsTable(DELTA_TABLE)
)

ingest_elapsed = time.time() - ingest_start
rows_per_sec = source_count / ingest_elapsed if ingest_elapsed > 0 else 0

# Verify
delta_df = spark.table(DELTA_TABLE)
delta_count = delta_df.count()
INITIAL_COLS = delta_df.columns

print(f"\nDelta table created: {DELTA_TABLE}")
print(f"  Rows          : {delta_count:,}")
print(f"  Columns       : {len(INITIAL_COLS)} ({source_cols} source + {len(PARTITION_COLS)} partition)")
print(f"  Partitions    : {PARTITION_COLS}")
print(f"  Time          : {ingest_elapsed:.1f}s")
print(f"  Throughput    : {rows_per_sec:,.0f} rows/sec")
print(f"\nSchema:")
delta_df.printSchema()

In [ ]:
# ---------------------------------------------------------------------------
# Initial SQL Endpoint Sync
# ---------------------------------------------------------------------------
# After creating the Delta table, we trigger an initial SQL endpoint sync
# so the base table (32 columns) is visible before we begin EDA.
# The full sync function is defined in Phase 6. For now, we define a
# lightweight version that we will replace later.

import sempy.fabric as fabric

def _quick_sync():
    """Trigger a SQL endpoint metadata refresh (lightweight version)."""
    client = fabric.FabricRestClient()
    workspace_id = fabric.get_workspace_id()
    lakehouse_id = fabric.get_lakehouse_id()

    # Get SQL endpoint ID
    lh_resp = client.get(f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    lh_resp.raise_for_status()
    sql_endpoint_id = lh_resp.json()["properties"]["sqlEndpointProperties"]["id"]

    # Trigger refresh
    url = f"/v1/workspaces/{workspace_id}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata"
    payload = {
        "recreateTables": False,
        "timeout": {"timeUnit": "Minutes", "value": 15},
    }
    resp = client.post(url, json=payload)

    if resp.status_code == 200:
        print("Initial sync completed (synchronous).")
        return

    if resp.status_code == 202:
        op_id = resp.headers.get("x-ms-operation-id", "")
        print(f"Initial sync started (async). Operation: {op_id}")
        for _ in range(120):
            time.sleep(5)
            poll = client.get(f"/v1/operations/{op_id}")
            status = poll.json().get("status", "Unknown")
            if status == "Succeeded":
                print("Initial sync completed.")
                return
            if status == "Failed":
                print(f"Initial sync failed: {poll.json()}")
                return
        print("Initial sync timed out after 10 minutes.")
    else:
        print(f"Unexpected response: {resp.status_code} - {resp.text[:200]}")

_quick_sync()

---
## Phase 2: Exploratory Data Analysis

Before engineering new features, we explore the dataset to understand its shape, quality, and distributions. This is standard practice before any transformation work, and it doubles as content for the blog post.

Key areas:
- **Data quality**: null counts per column, data type verification
- **Business distributions**: booking types, destinations, payment methods
- **Financial analysis**: revenue by segment, membership tier impact
- **Temporal patterns**: monthly booking trends, seasonal effects

In [ ]:
# ---------------------------------------------------------------------------
# EDA: Data Quality and Null Analysis
# ---------------------------------------------------------------------------
df = spark.table(DELTA_TABLE)

print(f"Dataset: {df.count():,} rows x {len(df.columns)} columns\n")

# Null counts per column
print("Null Analysis:")
print("-" * 50)
null_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]
null_counts = df.select(null_exprs).collect()[0]
total_rows = df.count()

for col_name in df.columns:
    null_count = null_counts[col_name]
    if null_count > 0:
        pct = (null_count / total_rows) * 100
        print(f"  {col_name:<30s} {null_count:>12,} nulls ({pct:.1f}%)")

print(f"\nColumns with no nulls: {sum(1 for c in df.columns if null_counts[c] == 0)}/{len(df.columns)}")

# Booking type distribution
print("\n\nBooking Type Distribution:")
print("-" * 50)
display(
    df.groupBy("booking_type")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / total_rows * 100, 1).alias("pct"),
        F.round(F.avg("total_amount"), 2).alias("avg_amount"),
    )
    .orderBy(F.desc("count"))
)

# Top 10 destination countries
print("\n\nTop 10 Destination Countries:")
print("-" * 50)
display(
    df.groupBy("destination_country")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.sum("total_amount"), 0).alias("total_revenue"),
    )
    .orderBy(F.desc("bookings"))
    .limit(10)
)

In [ ]:
# ---------------------------------------------------------------------------
# EDA: Revenue Analysis and Trends
# ---------------------------------------------------------------------------

# Revenue by booking type
print("Revenue by Booking Type:")
print("-" * 50)
display(
    df.groupBy("booking_type")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discounts"),
        F.round(F.sum("tax_amount"), 2).alias("total_tax"),
    )
    .orderBy(F.desc("total_revenue"))
)

# Monthly booking trend
print("\n\nMonthly Booking Trend (last 12 months):")
print("-" * 50)
display(
    df.withColumn("year_month", F.date_format("booking_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.sum("total_amount"), 0).alias("revenue"),
    )
    .orderBy(F.desc("year_month"))
    .limit(12)
)

# Payment method distribution
print("\n\nPayment Method Distribution:")
print("-" * 50)
display(
    df.groupBy("payment_method")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("total_amount"), 2).alias("avg_spend"),
    )
    .orderBy(F.desc("count"))
)

# Membership tier vs average spend
print("\n\nMembership Tier vs Average Spend:")
print("-" * 50)
display(
    df.groupBy("user_membership_tier")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.avg("total_amount"), 2).alias("avg_spend"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.round(F.sum(F.when(F.col("is_repeat_booking"), 1).otherwise(0)) / F.count("*") * 100, 1).alias("repeat_pct"),
    )
    .orderBy(
        F.when(F.col("user_membership_tier") == "Diamond", 1)
        .when(F.col("user_membership_tier") == "Platinum", 2)
        .when(F.col("user_membership_tier") == "Gold", 3)
        .when(F.col("user_membership_tier") == "Silver", 4)
        .otherwise(5)
    )
)

# Booking status breakdown
print("\n\nBooking Status Distribution:")
print("-" * 50)
display(
    df.groupBy("booking_status")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / total_rows * 100, 1).alias("pct"),
    )
    .orderBy(F.desc("count"))
)

---
## Phase 3: Feature Engineering

This is the step that creates the **schema change scenario**. We will add 10 new calculated columns to the Delta table, bringing the total from 32 to 42 columns.

These columns are the kind of transformations that Analytics Engineers typically push upstream into dbt models or PySpark notebooks, rather than calculating them in DAX or SQL views:

| New Column | Logic |
|---|---|
| `booking_quarter` | Quarter of the year (1 to 4) |
| `booking_day_of_week` | Day of week (1=Sunday through 7=Saturday) |
| `is_weekend_booking` | True if booked on Saturday or Sunday |
| `trip_category` | "domestic" if user and destination country match, else "international" |
| `spend_tier` | budget / mid_range / premium / luxury based on total_amount thresholds |
| `discount_percentage` | Discount as a percentage of base amount |
| `net_revenue` | Total amount for completed bookings, 0 for cancelled/refunded |
| `is_rated` | True if the booking has a rating |
| `rating_bucket` | poor / average / good / excellent based on rating value |
| `days_since_booking` | Days between booking date and current date |

Note: `booking_year` and `booking_month` were already added during ingestion as partition columns.

In [ ]:
# ---------------------------------------------------------------------------
# Phase 3: Feature Engineering (10 new columns)
# ---------------------------------------------------------------------------
fe_start = time.time()

df = spark.table(DELTA_TABLE)
cols_before = len(df.columns)
print(f"Columns before feature engineering: {cols_before}")

df_enriched = (
    df
    # Time-based features
    .withColumn("booking_quarter", F.quarter("booking_date"))
    .withColumn("booking_day_of_week", F.dayofweek("booking_date"))
    .withColumn("is_weekend_booking",
        F.when(F.dayofweek("booking_date").isin(1, 7), True).otherwise(False)
    )
    # Trip classification
    .withColumn("trip_category",
        F.when(F.col("user_country") == F.col("destination_country"), "domestic")
        .otherwise("international")
    )
    # Financial features
    .withColumn("spend_tier",
        F.when(F.col("total_amount") < 100, "budget")
        .when(F.col("total_amount") < 500, "mid_range")
        .when(F.col("total_amount") < 1500, "premium")
        .otherwise("luxury")
    )
    .withColumn("discount_percentage",
        F.when(F.col("base_amount") > 0,
            F.round(F.col("discount_amount") / F.col("base_amount") * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("net_revenue",
        F.when(F.col("booking_status") == "completed", F.col("total_amount"))
        .otherwise(F.lit(0.0))
    )
    # Engagement features
    .withColumn("is_rated", F.col("rating").isNotNull())
    .withColumn("rating_bucket",
        F.when(F.col("rating").isNull(), None)
        .when(F.col("rating") <= 2.0, "poor")
        .when(F.col("rating") <= 3.0, "average")
        .when(F.col("rating") <= 4.0, "good")
        .otherwise("excellent")
    )
    # Recency
    .withColumn("days_since_booking",
        F.datediff(F.current_date(), F.col("booking_date"))
    )
)

# Overwrite Delta table with new schema
(
    df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy(*PARTITION_COLS)
    .saveAsTable(DELTA_TABLE)
)

fe_elapsed = time.time() - fe_start

# Verify
df_final = spark.table(DELTA_TABLE)
cols_after = len(df_final.columns)
new_columns = set(df_final.columns) - set(INITIAL_COLS)

print(f"\nColumns after feature engineering: {cols_after}")
print(f"New columns added: {len(new_columns)}")
print(f"  {sorted(new_columns)}")
print(f"Time: {fe_elapsed:.1f}s")

---
## Phase 4: The Problem. SQL Endpoint Schema Mismatch

At this point, our Delta table has **42 columns**. But if you open the SQL analytics endpoint in the Fabric portal and run:

```sql
SELECT * FROM voyagehub_transactions
```

You will only see **32 columns**. The 10 new feature columns are invisible.

This is because the SQL analytics endpoint maintains its own metadata cache. When Spark writes new columns to the Delta table, the endpoint does not automatically refresh its schema. You have to either:

1. **Wait** for the automatic background sync (unreliable, can take minutes to hours)
2. **Click "Refresh Schema"** in the Fabric UI (manual, slow, not automatable)
3. **Use the REST API** to force a refresh (what we will do next)

For production pipelines, option 3 is the only viable approach. You cannot ask your Data Pipeline to click a button.

In [ ]:
# ---------------------------------------------------------------------------
# Demonstrate the Schema Mismatch
# ---------------------------------------------------------------------------

# The Delta table has 42 columns
delta_cols = spark.table(DELTA_TABLE).columns
print(f"Delta table column count: {len(delta_cols)}")
print(f"Delta table columns:")
for i, col in enumerate(delta_cols, 1):
    marker = " [NEW]" if col in new_columns else ""
    print(f"  {i:2d}. {col}{marker}")

print(f"\n{'=' * 70}")
print("  THE PROBLEM")
print("=" * 70)
print(f"  Delta table has {len(delta_cols)} columns.")
print(f"  SQL analytics endpoint still shows {len(INITIAL_COLS)} columns.")
print(f"  Missing columns: {sorted(new_columns)}")
print(f"\n  These columns are invisible to SQL queries, Power BI Direct Lake")
print(f"  models, and any tool that reads from the SQL endpoint.")
print("=" * 70)

---
## Phase 5: The Solution. Programmatic SQL Endpoint Refresh

The Fabric REST API provides an endpoint for on-demand metadata refresh:

```
POST /v1/workspaces/{workspaceId}/sqlEndpoints/{sqlEndpointId}/refreshMetadata
```

**How it works:**

1. **Authentication**: `sempy.fabric.FabricRestClient()` handles token management automatically inside Fabric notebooks
2. **Get SQL endpoint ID**: Query the Lakehouse metadata via `GET /v1/workspaces/{ws}/lakehouses/{lh}` and extract `properties.sqlEndpointProperties.id`
3. **Trigger refresh**: POST to the refresh endpoint with configuration (recreate tables, timeout)
4. **Handle response**: The API returns either HTTP 200 (synchronous completion) or HTTP 202 (asynchronous Long Running Operation)
5. **Poll LRO**: For 202 responses, poll `GET /v1/operations/{operationId}` until the status is `Succeeded` or `Failed`
6. **Get results**: Fetch per-table sync status from `GET /v1/operations/{operationId}/result`

**Reliability features built into our implementation:**
- Retry on HTTP 429 (rate limit) with `Retry-After` header
- Retry on HTTP 5xx (server errors) with exponential backoff
- Configurable timeout and maximum retry count
- Per-table status reporting with error details

In [ ]:
# ---------------------------------------------------------------------------
# Phase 5: Reusable SQL Endpoint Sync Function
# ---------------------------------------------------------------------------
import sempy.fabric as fabric


def get_sql_endpoint_id(
    client,
    workspace_id: str,
    lakehouse_id: str,
) -> str:
    """Retrieve the SQL analytics endpoint ID for a given Lakehouse."""
    response = client.get(f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    response.raise_for_status()

    lh_info = response.json()
    sql_props = lh_info.get("properties", {}).get("sqlEndpointProperties", {})
    sql_endpoint_id = sql_props.get("id")
    provisioning_status = sql_props.get("provisioningStatus", "Unknown")

    if not sql_endpoint_id:
        raise ValueError(
            f"No SQL endpoint found for Lakehouse '{lakehouse_id}'. "
            f"Provisioning status: {provisioning_status}"
        )

    logger.info(f"SQL Endpoint ID: {sql_endpoint_id} (status: {provisioning_status})")
    return sql_endpoint_id


def _extract_operation_id(headers: dict) -> str:
    """Extract operation ID from response headers (multiple strategies)."""
    op_id = headers.get("x-ms-operation-id")
    if op_id:
        return op_id

    location = headers.get("Location") or headers.get("location", "")
    match = re.search(r"/operations/([0-9a-fA-F-]+)", location)
    if match:
        return match.group(1)

    return ""


def sync_sql_endpoint(
    workspace_id: str = None,
    lakehouse_id: str = None,
    recreate_tables: bool = False,
    timeout_minutes: int = 15,
    poll_interval: int = 5,
    max_poll_attempts: int = 180,
    max_retries: int = 3,
) -> dict:
    """
    Programmatically refresh the SQL Analytics Endpoint schema.

    Uses the official Fabric REST API to trigger a metadata sync, polls
    for completion via the Long Running Operation (LRO) pattern, and
    returns per-table sync status.

    Parameters
    ----------
    workspace_id : str, optional
        Override workspace ID. Defaults to current notebook context.
    lakehouse_id : str, optional
        Override lakehouse ID. Defaults to current notebook context.
    recreate_tables : bool
        If True, drops and recreates ALL tables on the SQL endpoint.
        Use only for resolving persistent inconsistencies.
    timeout_minutes : int
        How long Fabric will attempt the sync before timing out.
    poll_interval : int
        Seconds between LRO status checks.
    max_poll_attempts : int
        Safety cap on polling iterations.
    max_retries : int
        Retry count for 429/5xx errors.

    Returns
    -------
    dict with keys: status, tables, elapsed_seconds, attempts, raw_response
    """
    client = fabric.FabricRestClient()

    if workspace_id is None:
        workspace_id = fabric.get_workspace_id()
    if lakehouse_id is None:
        lakehouse_id = fabric.get_lakehouse_id()

    sql_endpoint_id = get_sql_endpoint_id(client, workspace_id, lakehouse_id)

    url = f"/v1/workspaces/{workspace_id}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata"
    payload = {
        "recreateTables": recreate_tables,
        "timeout": {
            "timeUnit": "Minutes",
            "value": timeout_minutes,
        },
    }

    result = {
        "start_time": datetime.now(timezone.utc).isoformat(),
        "workspace_id": workspace_id,
        "lakehouse_id": lakehouse_id,
        "sql_endpoint_id": sql_endpoint_id,
        "status": "unknown",
        "tables": [],
        "attempts": 0,
        "elapsed_seconds": 0,
        "raw_response": None,
    }

    start_time = time.time()

    for attempt in range(1, max_retries + 1):
        result["attempts"] = attempt
        logger.info(f"[Attempt {attempt}/{max_retries}] Triggering SQL endpoint refresh...")

        try:
            response = client.post(url, json=payload)
        except Exception as e:
            logger.error(f"Request failed: {e}")
            if attempt < max_retries:
                wait = 2 ** attempt * 5
                logger.info(f"Retrying in {wait}s...")
                time.sleep(wait)
                continue
            result["status"] = "exception"
            result["error"] = str(e)
            break

        # --- HTTP 200: Synchronous completion ---
        if response.status_code == 200:
            logger.info("Sync completed synchronously (HTTP 200).")
            resp_data = response.json() if response.text else {}
            result["status"] = "Succeeded"
            result["tables"] = resp_data.get("value", [])
            result["raw_response"] = resp_data
            break

        # --- HTTP 202: Async LRO ---
        if response.status_code == 202:
            operation_id = _extract_operation_id(dict(response.headers))
            retry_after = int(response.headers.get("Retry-After", poll_interval))

            if not operation_id:
                logger.error("202 returned but no operation ID found in headers.")
                result["status"] = "error"
                result["error"] = "No operation ID in 202 response headers"
                break

            logger.info(f"Async operation started. ID: {operation_id}")
            logger.info(f"Polling every {retry_after}s (max {max_poll_attempts} attempts)...")

            for poll_num in range(1, max_poll_attempts + 1):
                time.sleep(retry_after)

                try:
                    poll_resp = client.get(f"/v1/operations/{operation_id}")
                    poll_resp.raise_for_status()
                    poll_data = poll_resp.json()
                except Exception as e:
                    logger.warning(f"Poll #{poll_num} failed: {e}")
                    continue

                status = poll_data.get("status", "Unknown")
                pct = poll_data.get("percentComplete", "?")

                if poll_num % 5 == 0 or status not in ("Running", "NotStarted"):
                    elapsed = round(time.time() - start_time, 1)
                    logger.info(f"  Poll #{poll_num}: status={status}, progress={pct}%, elapsed={elapsed}s")

                if status == "Succeeded":
                    elapsed = round(time.time() - start_time, 1)
                    logger.info(f"Sync completed in {elapsed}s.")

                    # Fetch per-table results
                    try:
                        result_resp = client.get(f"/v1/operations/{operation_id}/result")
                        result_resp.raise_for_status()
                        result_data = result_resp.json()
                    except Exception:
                        result_data = poll_data

                    result["status"] = "Succeeded"
                    result["tables"] = result_data.get("value", [])
                    result["raw_response"] = result_data
                    break

                if status == "Failed":
                    elapsed = round(time.time() - start_time, 1)
                    error_info = poll_data.get("error", {})
                    logger.error(f"Sync FAILED after {elapsed}s: {json.dumps(error_info, indent=2)}")
                    result["status"] = "Failed"
                    result["error"] = error_info
                    result["raw_response"] = poll_data
                    break

                # Respect Retry-After from poll response if present
                new_retry = poll_resp.headers.get("Retry-After")
                if new_retry and str(new_retry).isdigit():
                    retry_after = int(new_retry)

            else:
                logger.warning(f"Polling timed out after {max_poll_attempts} attempts.")
                result["status"] = "PollingTimeout"
                result["raw_response"] = poll_data

            break  # Exit retry loop after handling 202

        # --- HTTP 429: Rate Limited ---
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 2 ** attempt * 10))
            logger.warning(f"Rate limited (429). Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue

        # --- HTTP 5xx: Server Error ---
        if response.status_code >= 500:
            wait = 2 ** attempt * 5
            logger.warning(f"Server error ({response.status_code}). Retrying in {wait}s...")
            time.sleep(wait)
            continue

        # --- Unexpected status ---
        logger.error(f"Unexpected response: {response.status_code}")
        result["status"] = f"HTTP_{response.status_code}"
        result["error"] = response.text[:500] if response.text else "No response body"
        break

    result["end_time"] = datetime.now(timezone.utc).isoformat()
    result["elapsed_seconds"] = round(time.time() - start_time, 2)

    return result


def build_sync_report(tables: list) -> pd.DataFrame:
    """Build a pandas DataFrame summarizing per-table sync results."""
    if not tables:
        return pd.DataFrame()

    rows = []
    for t in tables:
        error = t.get("error", {}) or {}
        rows.append({
            "Table": t.get("tableName", "?"),
            "Status": t.get("status", "?"),
            "Last Sync": t.get("lastSuccessfulSyncDateTime", "N/A"),
            "Start": t.get("startDateTime", "N/A"),
            "End": t.get("endDateTime", "N/A"),
            "Error Code": error.get("errorCode", ""),
            "Error Message": error.get("message", ""),
        })

    df = pd.DataFrame(rows)
    status_order = {"Failure": 0, "Success": 1, "NotRun": 2}
    df["_sort"] = df["Status"].map(status_order).fillna(3)
    return df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)


print("sync_sql_endpoint() and build_sync_report() defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Execute the SQL Endpoint Refresh
# ---------------------------------------------------------------------------
print("Triggering SQL endpoint schema refresh...")
print("=" * 70)

sync_result = sync_sql_endpoint(
    recreate_tables=False,
    timeout_minutes=15,
    poll_interval=5,
    max_poll_attempts=180,
    max_retries=3,
)

print("\n" + "=" * 70)
print("  SQL ENDPOINT SYNC RESULT")
print("=" * 70)
print(f"  Status     : {sync_result['status']}")
print(f"  Duration   : {sync_result['elapsed_seconds']}s")
print(f"  Attempts   : {sync_result['attempts']}")
if sync_result.get("error"):
    print(f"  Error      : {sync_result['error']}")
print("=" * 70)

# Per-table report
if sync_result["tables"]:
    report_df = build_sync_report(sync_result["tables"])
    counts = report_df["Status"].value_counts().to_dict()
    print(f"\n  Total tables : {len(report_df)}")
    print(f"  Synced       : {counts.get('Success', 0)}")
    print(f"  Already OK   : {counts.get('NotRun', 0)}")
    print(f"  Failed       : {counts.get('Failure', 0)}")
    print()
    display(report_df)
else:
    print("\nNo per-table results returned.")

---
## Phase 6: Verification

After the sync completes, we verify that all 42 columns are now visible through the SQL analytics endpoint. We also run sample queries using the newly added feature columns to confirm they are queryable.

In [ ]:
# ---------------------------------------------------------------------------
# Phase 6: Verification
# ---------------------------------------------------------------------------

# Verify column count
verify_df = spark.table(DELTA_TABLE)
final_cols = verify_df.columns
expected_col_count = 42

print("=" * 70)
print("  VERIFICATION")
print("=" * 70)
print(f"  Initial columns  : {len(INITIAL_COLS)}")
print(f"  Final columns    : {len(final_cols)}")
print(f"  Expected         : {expected_col_count}")
print(f"  Match            : {'YES' if len(final_cols) == expected_col_count else 'NO'}")
print("=" * 70)

# Sample query: Spend tier distribution
print("\n\nSpend Tier Distribution:")
print("-" * 50)
display(
    verify_df.groupBy("spend_tier")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.avg("total_amount"), 2).alias("avg_amount"),
        F.round(F.avg("discount_percentage"), 2).alias("avg_discount_pct"),
    )
    .orderBy(
        F.when(F.col("spend_tier") == "budget", 1)
        .when(F.col("spend_tier") == "mid_range", 2)
        .when(F.col("spend_tier") == "premium", 3)
        .otherwise(4)
    )
)

# Sample query: Trip category breakdown
print("\n\nTrip Category (Domestic vs International):")
print("-" * 50)
display(
    verify_df.groupBy("trip_category")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.sum("net_revenue"), 0).alias("net_revenue"),
        F.round(F.avg("days_since_booking"), 0).alias("avg_days_since"),
    )
    .orderBy("trip_category")
)

# Sample query: Rating bucket analysis
print("\n\nRating Bucket Analysis:")
print("-" * 50)
display(
    verify_df.groupBy("rating_bucket")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.avg("total_amount"), 2).alias("avg_spend"),
        F.sum(F.when(F.col("is_repeat_booking"), 1).otherwise(0)).alias("repeat_bookings"),
    )
    .orderBy(
        F.when(F.col("rating_bucket").isNull(), 5)
        .when(F.col("rating_bucket") == "poor", 1)
        .when(F.col("rating_bucket") == "average", 2)
        .when(F.col("rating_bucket") == "good", 3)
        .otherwise(4)
    )
)

# Weekend vs weekday
print("\n\nWeekend vs Weekday Bookings:")
print("-" * 50)
display(
    verify_df.groupBy("is_weekend_booking")
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.avg("total_amount"), 2).alias("avg_amount"),
        F.round(F.avg("session_duration_seconds"), 0).alias("avg_session_sec"),
    )
    .orderBy("is_weekend_booking")
)

---
## Phase 7: Conclusion and Reusable Utility

### What We Demonstrated

1. **Loaded 60M rows** of synthetic travel data into a partitioned Delta table
2. **Performed feature engineering** that added 10 new calculated columns
3. **Identified the problem**: the SQL analytics endpoint did not reflect the new schema
4. **Solved it programmatically** using the official Fabric REST API
5. **Verified** that all 42 columns are now queryable through the SQL endpoint

### The Reusable Pattern

The `sync_sql_endpoint()` function defined in Phase 5 can be copied into any Fabric notebook and called after any Spark write operation:

```python
# After writing to a Delta table
df.write.format("delta").mode("overwrite").saveAsTable("my_table")

# Force the SQL endpoint to pick up the changes
result = sync_sql_endpoint()
assert result["status"] == "Succeeded"
```

### Pipeline Integration

To integrate this into a Fabric Data Pipeline, wrap the sync in a notebook activity:

```python
# In your pipeline notebook (called after the transformation notebook)
import sempy.fabric as fabric
# ... paste the sync_sql_endpoint() function ...
result = sync_sql_endpoint(timeout_minutes=20)
if result["status"] != "Succeeded":
    raise RuntimeError(f"SQL endpoint sync failed: {result}")
```

### Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| HTTP 401 Unauthorized | Token expired or insufficient permissions | Re-run the notebook (token refreshes automatically) |
| HTTP 429 Too Many Requests | API rate limit exceeded | Built-in retry logic handles this automatically |
| HTTP 5xx Server Error | Transient Fabric service issue | Built-in exponential backoff handles this |
| Polling timeout after 15 min | Very large schema change or service slowness | Increase `timeout_minutes` and `max_poll_attempts` |
| Columns still missing after sync | Endpoint cache not yet propagated | Wait 1 to 2 minutes, then refresh the portal browser tab |
| `Failure` status on a table | Unsupported column types (ARRAY, MAP, STRUCT) or naming conflicts | Check error details in the sync report |
| `NotRun` status on a table | Table was already in sync | This is normal and expected for unchanged tables |

### Official Documentation

- [Refresh SQL Endpoint Metadata API](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/refresh-sql-endpoint-metadata)
- [Get Lakehouse API](https://learn.microsoft.com/en-us/rest/api/fabric/lakehouse/items/get-lakehouse)
- [Long Running Operations](https://learn.microsoft.com/en-us/rest/api/fabric/articles/long-running-operation)
- [SQL Analytics Endpoint Overview](https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-sql-analytics-endpoint)
- [FabricRestClient Reference](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric.fabricrestclient)

In [ ]:
# ---------------------------------------------------------------------------
# Final Run Summary
# ---------------------------------------------------------------------------
RUN_END = datetime.now(timezone.utc)
total_runtime = (RUN_END - RUN_START).total_seconds()

print("=" * 70)
print("  NOTEBOOK RUN COMPLETE")
print("=" * 70)
print(f"  Run ID              : {RUN_ID}")
print(f"  Dataset             : {delta_count:,} rows")
print(f"  Initial columns     : {len(INITIAL_COLS)} (30 source + 2 partition)")
print(f"  Final columns       : {len(final_cols)} (+ 10 features)")
print(f"  SQL Endpoint Sync   : {sync_result['status']} in {sync_result['elapsed_seconds']}s")
print(f"  Ingestion time      : {ingest_elapsed:.1f}s")
print(f"  Feature eng. time   : {fe_elapsed:.1f}s")
print(f"  Total notebook time : {total_runtime:.1f}s ({total_runtime/60:.1f} min)")
print(f"  Completed at        : {RUN_END.isoformat()}")
print("=" * 70)